# TF-IDF Enhanced Training (Hybrid Approach)

**Combines your 22 handcrafted features with TF-IDF for better semantic understanding**

## Why TF-IDF + Your Features?

### **Your Current Features (22 total):**
- ✅ **Fast inference** (22 features)
- ✅ **Interpretable** (understandable features)
- ✅ **Domain-specific** (tailored for duplicate detection)
- ❌ **Limited semantic understanding** (no word importance)

### **TF-IDF Features:**
- ✅ **Semantic similarity** (word importance & rarity)
- ✅ **N-gram support** (phrase matching)
- ✅ **Automatic weighting** (common words downweighted)
- ❌ **High dimensionality** (1000+ features)
- ❌ **Slower inference**
- ❌ **Less interpretable**

### **Hybrid Approach (BEST SOLUTION):**
- **22 handcrafted + 500-1000 TF-IDF features**
- **Memory efficient** (fits your 16GB RAM)
- **Better performance** (+2-5% F1 score)
- **Balanced speed/accuracy**

**Expected Result:** F1 score 0.77-0.82 (vs 0.75-0.80 with features only)

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
import pickle
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


## 1. Load Dataset

In [2]:
# Load and sample dataset
df = pd.read_csv('train.csv')
df_sample = df.sample(100000, random_state=42)

## 2. Create Hybrid Features (Your 22 + TF-IDF)

In [3]:
# Import your existing feature functions
import sys
sys.path.append('.')
from helper import preprocess, test_fetch_token_features, test_fetch_length_features, test_fetch_fuzzy_features
from helper import test_common_words, test_total_words

def create_hybrid_features(df_subset, tfidf_vectorizer=None, fit_tfidf=True):
    """Create hybrid features: your 22 handcrafted + TF-IDF"""
    
    # 1. Create your 22 handcrafted features
    print("Creating handcrafted features...")
    handcrafted_features = []
    target = []
    
    for idx, row in df_subset.iterrows():
        q1 = str(row['question1'])
        q2 = str(row['question2'])
        
        # Preprocess
        q1 = preprocess(q1)
        q2 = preprocess(q2)
        
        # Your 22 features
        feature_list = []
        feature_list.append(len(q1))
        feature_list.append(len(q2))
        feature_list.append(len(q1.split(" ")))
        feature_list.append(len(q2.split(" ")))
        feature_list.append(test_common_words(q1, q2))
        feature_list.append(test_total_words(q1, q2))
        feature_list.append(round(test_common_words(q1, q2) / (test_total_words(q1, q2) + 0.0001), 2))
        
        feature_list.extend(test_fetch_token_features(q1, q2))
        feature_list.extend(test_fetch_length_features(q1, q2))
        feature_list.extend(test_fetch_fuzzy_features(q1, q2))
        
        handcrafted_features.append(feature_list)
        target.append(row['is_duplicate'])
    
    handcrafted_features = np.array(handcrafted_features)
    target = np.array(target)
    
    # 2. Create TF-IDF features
    print("Creating TF-IDF features...")
    
    # Combine questions for TF-IDF (capture similarity in combined space)
    combined_questions = df_subset['question1'] + " " + df_subset['question2']
    
    if fit_tfidf:
        # Fit TF-IDF on training data
        tfidf_vectorizer = TfidfVectorizer(
            max_features=500,  # Limit for memory (can increase to 1000)
            ngram_range=(1, 2),  # Unigrams and bigrams
            stop_words='english',  # Remove stop words
            min_df=5,  # Word must appear in at least 5 questions
            max_df=0.8  # Word can appear in max 80% of questions
        )
        tfidf_features = tfidf_vectorizer.fit_transform(combined_questions)
    else:
        # Transform test data
        tfidf_features = tfidf_vectorizer.transform(combined_questions)
    
    # 3. Combine features
    print("Combining features...")
    
    # Convert handcrafted to sparse matrix for efficient combination
    handcrafted_sparse = csr_matrix(handcrafted_features)
    
    # Horizontally stack features
    combined_features = hstack([handcrafted_sparse, tfidf_features])
    
    print(f"Handcrafted features: {handcrafted_features.shape[1]}")
    print(f"TF-IDF features: {tfidf_features.shape[1]}")
    print(f"Total features: {combined_features.shape[1]}")
    print(f"Memory usage: ~{combined_features.data.nbytes / 1024 / 1024:.1f} MB")
    
    return combined_features, target, tfidf_vectorizer

print("Hybrid feature creation function ready")

Hybrid feature creation function ready


In [4]:
# Split data first (important for TF-IDF fitting)
train_df, test_df = train_test_split(
    df_sample, test_size=0.3, random_state=42, stratify=df_sample['is_duplicate']
)

print(f"Train set: {len(train_df)} samples")
print(f"Test set: {len(test_df)} samples")

# Create features for training set (fit TF-IDF)
print("\nCreating training features...")
X_train_combined, y_train, tfidf_vectorizer = create_hybrid_features(train_df, fit_tfidf=True)

# Create features for test set (transform only)
print("\nCreating test features...")
X_test_combined, y_test, _ = create_hybrid_features(test_df, tfidf_vectorizer=tfidf_vectorizer, fit_tfidf=False)

print(f"\nFinal shapes:")
print(f"X_train: {X_train_combined.shape}")
print(f"X_test: {X_test_combined.shape}")
print(f"Feature types: {type(X_train_combined)}")

Train set: 70000 samples
Test set: 30000 samples

Creating training features...
Creating handcrafted features...
Creating TF-IDF features...
Combining features...
Handcrafted features: 22
TF-IDF features: 500
Total features: 522
Memory usage: ~12.5 MB

Creating test features...
Creating handcrafted features...
Creating TF-IDF features...
Combining features...
Handcrafted features: 22
TF-IDF features: 500
Total features: 522
Memory usage: ~5.4 MB

Final shapes:
X_train: (70000, 522)
X_test: (30000, 522)
Feature types: <class 'scipy.sparse._csr.csr_matrix'>


## 3. Train Model with Hybrid Features

In [5]:
# Use Gradient Boosting (handles sparse matrices well)
print("Training Gradient Boosting with hybrid features...")
model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    min_samples_split=20,
    subsample=0.8,
    random_state=42,
    verbose=1
)

# Convert sparse to dense for Gradient Boosting (it doesn't handle sparse well)
# Note: This increases memory usage but necessary for GB
print("Converting sparse to dense (required for Gradient Boosting)...")
X_train_dense = X_train_combined.toarray()
X_test_dense = X_test_combined.toarray()

print(f"Dense shapes: {X_train_dense.shape}, {X_test_dense.shape}")
print(f"Memory for dense: ~{(X_train_dense.nbytes + X_test_dense.nbytes) / 1024 / 1024:.1f} MB")

# Quick cross-validation
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train_dense, y_train, cv=skf, scoring='f1')
print(f"\nCross-validation F1 scores: {cv_scores}")
print(f"Mean CV F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Train final model
model.fit(X_train_dense, y_train)
print("\nModel training completed!")

Training Gradient Boosting with hybrid features...
Converting sparse to dense (required for Gradient Boosting)...
Dense shapes: (70000, 522), (30000, 522)
Memory for dense: ~398.3 MB
      Iter       Train Loss      OOB Improve   Remaining Time 
         1           1.2630           0.0561            1.81m
         2           1.2142           0.0428            1.70m
         3           1.1740           0.0375            1.66m
         4           1.1407           0.0349            1.66m
         5           1.1118           0.0281            1.63m
         6           1.0866           0.0250            1.61m
         7           1.0639           0.0177            1.59m
         8           1.0448           0.0179            1.56m
         9           1.0313           0.0273            1.55m
        10           1.0133           0.0046            1.52m
        20           0.9303           0.0117            1.38m
        30           0.8883          -0.0003            1.21m
        40

## 4. Evaluate Hybrid Model

In [ ]:
# Predictions
y_pred = model.predict(X_test_dense)
y_pred_proba = model.predict_proba(X_test_dense)[:, 1]

# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print("="*60)
print("HYBRID MODEL RESULTS (22 Features + 500 TF-IDF)")
print("="*60)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {auc:.4f}")
print(f"\nCross-validation F1: {cv_scores.mean():.4f}")

HYBRID MODEL RESULTS (22 Features + 500 TF-IDF)
Accuracy:  0.7842
Precision: 0.7128
Recall:    0.7011
F1 Score:  0.7069
ROC-AUC:   0.8713

Cross-validation F1: 0.7039


In [21]:
confusion_matrix(y_test,y_pred)

array([[15718,  3146],
       [ 3329,  7807]])

## 5. Save Hybrid Model

In [11]:
# Save the hybrid model and vectorizer
with open('model_hybrid_tfidf.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

print("Hybrid model and TF-IDF vectorizer saved!")
print("Files: 'model_hybrid_tfidf.pkl', 'tfidf_vectorizer.pkl'")

Hybrid model and TF-IDF vectorizer saved!
Files: 'model_hybrid_tfidf.pkl', 'tfidf_vectorizer.pkl'
